In [1]:
import pandas as pd
df = pd.read_csv("../data/processed/after_encoding/df_encoded.csv")

In [2]:
import pandas as pd
import numpy as np

print("🔧 GENERATING NEW TARGETS...")
print("="*60)

# ==========================================
# TARGET 4: TIME TO TRACTION (time_to_1000_users_months)
# ==========================================
print("\n📊 Target 4: Time to 1000 Users")

# Check if column exists and has data
if 'time_to_1000_users_months' in df.columns:
    print(f"   Original column found!")
    print(f"   Non-null values: {df['time_to_1000_users_months'].notna().sum()}")
    print(f"   Range: {df['time_to_1000_users_months'].min():.1f} - {df['time_to_1000_users_months'].max():.1f} months")
    
    # Clean the target
    df['target_traction_time'] = df['time_to_1000_users_months'].clip(lower=0, upper=60)  # Cap at 5 years
    
else:
    print("   Column not found! Creating synthetic estimate...")
    # Estimate based on downloads growth
    df['target_traction_time'] = np.where(
        df['downloads'] >= 1000,
        np.maximum(1, df['age_months'] * 0.3),  # Estimate: reached 1k users in ~30% of lifetime
        np.nan
    )

print(f"   Final target stats:")
print(f"   Valid samples: {df['target_traction_time'].notna().sum()}")
print(f"   Mean: {df['target_traction_time'].mean():.1f} months")
print(f"   Median: {df['target_traction_time'].median():.1f} months")


# ==========================================
# TARGET 5: BREAK-EVEN TIME (break_even_time_months)
# ==========================================
print("\n📊 Target 5: Break-Even Time")

if 'break_even_time_months' in df.columns:
    print(f"   Original column found!")
    print(f"   Non-null values: {df['break_even_time_months'].notna().sum()}")
    print(f"   Range: {df['break_even_time_months'].min():.1f} - {df['break_even_time_months'].max():.1f} months")
    
    # Clean the target
    df['target_breakeven_time'] = df['break_even_time_months'].clip(lower=0, upper=84)  # Cap at 7 years
    
else:
    print("   Column not found! Creating estimate from profitability...")
    # Estimate: if profitable now, when did they break even?
    # If not profitable, estimate based on burn/revenue ratio
    df['target_breakeven_time'] = np.where(
        df['is_profitable'] == 1,
        df['age_months'] * 0.6,  # Assume broke even at 60% of current age
        np.nan  # Can't estimate for non-profitable
    )

print(f"   Final target stats:")
print(f"   Valid samples: {df['target_breakeven_time'].notna().sum()}")
print(f"   Mean: {df['target_breakeven_time'].mean():.1f} months")
print(f"   Median: {df['target_breakeven_time'].median():.1f} months")

print("\n✅ TARGETS GENERATED!")
print("="*60)


🔧 GENERATING NEW TARGETS...

📊 Target 4: Time to 1000 Users
   Column not found! Creating synthetic estimate...
   Final target stats:
   Valid samples: 44401
   Mean: 18.8 months
   Median: 14.3 months

📊 Target 5: Break-Even Time
   Column not found! Creating estimate from profitability...
   Final target stats:
   Valid samples: 701
   Mean: 34.0 months
   Median: 23.8 months

✅ TARGETS GENERATED!


In [4]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

print("="*60)
print("💰 MODEL 5: BREAK-EVEN TIME")
print("="*60)

# ==========================================
COMMON_LEAKS = [
    'success_label', 'still_operating', 'is_dead',
    'exit_status_Active', 'exit_status_Bankruptcy', 'exit_status_Failed',
    'exit_status_Operating', 'exit_status_Shut Down', 'exit_status_Shutdown',
    'exit_status_Still Active', 'exit_status_Unknown', 'exit_status_unknown',
    'failure_reason_cat_business_model', 'failure_reason_cat_competition',
    'failure_reason_cat_funding', 'failure_reason_cat_legal',
    'failure_reason_cat_market_fit', 'failure_reason_cat_multiple',
    'failure_reason_cat_no_market_fit', 'failure_reason_cat_other',
    'failure_reason_cat_ran_out_of_cash', 'failure_reason_cat_team',
    'failure_reason_cat_team_issues', 'failure_reason_cat_unknown',
    'failure_reason_cat_encoded',
    'success_reason_cat_revenue', 'success_reason_cat_unknown',
    'synthetic_data', 'data_source_synthetic', 'data_source_synthetic_augmentation'
]

TEXT_LEAKS = [c for c in df.columns if c.startswith('sf_') or c.startswith('fr_') or c.startswith('sr_')]
# ==========================================
BREAKEVEN_LEAKS = COMMON_LEAKS + TEXT_LEAKS + [
    # Target
    'break_even_time_months',
    'target_breakeven_time',
    'base_break_even_months',
    'estimated_break_even_months',
    
    # Profitability (directly related)
    'is_profitable',
    'monthly_profit_est',
    'revenue_to_burn_ratio',
    'capital_efficiency',
    
    # Financial metrics (outcome-dependent)
    'revenue_monthly', 'revenue_monthly_filled', 'revenue_estimated',
    'burn_rate_monthly_est',
    'remaining_investment',
    'estimated_runway_months',
    
    # Traction (comes after break-even usually)
    'downloads', 'downloads_per_month',
    'active_users',
    'upvotes', 'upvotes_per_month',
    'review_count', 'reviews_per_month',
    'engagement_score',
    'has_viral_reach',
    'rating_avg', 'rating_weighted', 'rating_confidence',
    'rating_tier', 'is_highly_rated', 'is_top_rated',
    'has_strong_reviews', 'has_strong_rating',
    'rating_vs_category',
    
    # Age (circular)
    'age_days', 'age_months', 'age_years',
    'is_young', 'is_established', 'is_old',
    'survival_months',
    
    # Derived scores
    'survival_score', 'growth_score', 'market_fit_score', 'risk_score',
    
    # Other model outputs
    'target_survival_months', 'target_growth_rate', 'target_growth_rate_log',
    'target_raise_funding', 'market_value_estimate', 'target_traction_time',
    
    # Metadata flags
    'has_marketing_channels', 'has_business_model', 'has_case_study_url',
    'has_brand', 'has_founders', 'has_website_url',
    'info_completeness', 'has_long_description', 'has_detailed_name',
    
    # Data source
    'data_source_failory', 'data_source_producthunt', 'data_source_indiehackers',
    'data_source_playstore', 'source_encoded',
    
    # Adjustments
    'rating_adjustment', 'team_adjustment', 'funding_adjustment',
    'funding_team_interaction', 'quality_traction_interaction',
    'maturity_quality_interaction', 'price_reach_interaction',
    
    # Unknown flags
    'business_model_unknown', 'product_type_unknown', 'main_category_unknown'
]

# ==========================================
# PREPARE DATA
# ==========================================
print("\n📊 Preparing data...")
X_be = df.drop(columns=BREAKEVEN_LEAKS, errors='ignore')
y_be = df['target_breakeven_time']

# Remove missing
mask = y_be.notna()
X_be = X_be[mask]
y_be = y_be[mask]

print(f"   Samples: {len(X_be)}")
print(f"   Features: {X_be.shape[1]}")
print(f"   Target range: {y_be.min():.1f} - {y_be.max():.1f} months")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_be, y_be, test_size=0.2, random_state=42
)

# ==========================================
# TRAIN MODEL
# ==========================================
print("\n🥊 Training XGBoost...")
be_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.02,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

be_model.fit(X_train, y_train)
print("✅ Training complete!")

# ==========================================
# EVALUATE
# ==========================================
y_pred = be_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n🏆 BREAK-EVEN TIME RESULTS:")
print(f"   R² Score: {r2:.4f}")
print(f"   MAE:      {mae:.2f} months")
print(f"   RMSE:     {rmse:.2f} months")

# Feature importance
imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': be_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(f"\n🔍 TOP 15 FEATURES:")
for i, row in imp.head(15).iterrows():
    print(f"   {row['Feature']:<35} {row['Importance']:.4f}")

# Decision
if r2 > 0.70:
    print(f"\n⚠️ R²={r2:.4f} - check for leaks")
elif r2 > 0.30:
    print(f"\n✅ GOOD! R²={r2:.4f} is realistic!")
    joblib.dump(be_model, '../models/breakeven_time_model_v1.pkl')
    print("✅ Saved: breakeven_time_model_v1.pkl")
else:
    print(f"\n⚠️ R²={r2:.4f} is low")
    joblib.dump(be_model, '../models/breakeven_time_model_v1.pkl')
    print("✅ Saved anyway")

print("\n" + "="*60)


💰 MODEL 5: BREAK-EVEN TIME

📊 Preparing data...
   Samples: 701
   Features: 188
   Target range: 7.2 - 114.5 months

🥊 Training XGBoost...
✅ Training complete!

🏆 BREAK-EVEN TIME RESULTS:
   R² Score: 0.4167
   MAE:      9.66 months
   RMSE:     18.27 months

🔍 TOP 15 FEATURES:
   is_free                             0.1442
   country_encoded                     0.1058
   content_rating                      0.0881
   sentiment_score                     0.0501
   description_length                  0.0442
   price_tier_numeric                  0.0378
   product_type_encoded                0.0314
   product_type_digital_app            0.0243
   market_saturation                   0.0211
   category_price_std                  0.0193
   category_type_encoded               0.0180
   num_founders                        0.0179
   name_length                         0.0171
   funding_per_month_alive             0.0168
   funding_per_round                   0.0158

✅ GOOD! R²=0.4167 is realisti

In [7]:
print("🔧 REGENERATING BREAK-EVEN TARGET WITH BETTER LOGIC...")

# Check what we have
print(f"is_profitable distribution:")
print(df['is_profitable'].value_counts())

# Better estimation strategy
df['target_breakeven_time_v2'] = np.nan

# Strategy 1: If we have the original column, use it
if 'break_even_time_months' in df.columns:
    df['target_breakeven_time_v2'] = df['break_even_time_months']
    print(f"Used original: {df['target_breakeven_time_v2'].notna().sum()} samples")

# Strategy 2: For profitable companies without data, estimate
if df['target_breakeven_time_v2'].notna().sum() < 5000:
    print("Supplementing with estimates...")
    
    # For profitable companies: estimate they broke even at 60% of current age
    profitable_mask = (df['is_profitable'] == 1) & (df['target_breakeven_time_v2'].isna())
    if 'age_months' in df.columns:
        df.loc[profitable_mask, 'target_breakeven_time_v2'] = df.loc[profitable_mask, 'age_months'] * 0.6
        print(f"   Added {profitable_mask.sum()} from profitable companies")
    
    # For high-revenue companies: estimate based on category average
    high_rev_mask = (df['revenue_monthly'] > 10000) & (df['target_breakeven_time_v2'].isna())
    if high_rev_mask.sum() > 0 and 'age_months' in df.columns:
        df.loc[high_rev_mask, 'target_breakeven_time_v2'] = df.loc[high_rev_mask, 'age_months'] * 0.5
        print(f"   Added {high_rev_mask.sum()} from high-revenue companies")

# Clip to reasonable range
df['target_breakeven_time_v2'] = df['target_breakeven_time_v2'].clip(lower=1, upper=84)

print(f"\n✅ Final samples: {df['target_breakeven_time_v2'].notna().sum()}")
print(f"   Mean: {df['target_breakeven_time_v2'].mean():.1f} months")
print(f"   Median: {df['target_breakeven_time_v2'].median():.1f} months")


🔧 REGENERATING BREAK-EVEN TARGET WITH BETTER LOGIC...
is_profitable distribution:
is_profitable
False    43892
True       701
Name: count, dtype: int64
Supplementing with estimates...
   Added 701 from profitable companies
   Added 42564 from high-revenue companies

✅ Final samples: 43265
   Mean: 31.1 months
   Median: 24.1 months


In [8]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

print("="*60)
print("💰 MODEL 5: BREAK-EVEN TIME (UPDATED)")
print("="*60)

# ==========================================
# PREPARE DATA WITH NEW TARGET
# ==========================================
print("\n📊 Preparing data...")
X_be = df.drop(columns=BREAKEVEN_LEAKS, errors='ignore')
y_be = df['target_breakeven_time_v2']  # ← Use NEW target!

# Remove missing
mask = y_be.notna()
X_be = X_be[mask]
y_be = y_be[mask]

print(f"   Samples: {len(X_be)} ✅")
print(f"   Features: {X_be.shape[1]}")
print(f"   Target range: {y_be.min():.1f} - {y_be.max():.1f} months")
print(f"   Target mean: {y_be.mean():.1f} months")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_be, y_be, test_size=0.2, random_state=42
)

# ==========================================
# TRAIN MODEL
# ==========================================
print("\n🥊 Training XGBoost...")
be_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.02,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

be_model.fit(X_train, y_train)
print("✅ Training complete!")

# ==========================================
# EVALUATE
# ==========================================
y_pred = be_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n🏆 BREAK-EVEN TIME RESULTS:")
print(f"   R² Score: {r2:.4f}")
print(f"   MAE:      {mae:.2f} months")
print(f"   RMSE:     {rmse:.2f} months")

# Feature importance
imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': be_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(f"\n🔍 TOP 15 FEATURES:")
for i, row in imp.head(15).iterrows():
    print(f"   {row['Feature']:<35} {row['Importance']:.4f}")

# ==========================================
# LEAK CHECK
# ==========================================
leak_check = ['revenue', 'burn', 'profit', 'age', 'download', 'rating']
found = []
for term in leak_check:
    leaked = [f for f in imp.head(20)['Feature'].values if term in f.lower()]
    if leaked:
        found.extend(leaked)
        print(f"\n⚠️ Potential leak '{term}': {leaked}")

if not found:
    print("\n✅ No obvious leaks in top 20!")

# ==========================================
# DECISION
# ==========================================
if r2 > 0.75:
    print(f"\n⚠️ R²={r2:.4f} - check for leaks above")
elif r2 > 0.35:
    print(f"\n✅ EXCELLENT! R²={r2:.4f} is production-ready!")
    joblib.dump(be_model, '../models/breakeven_time_model_v1.pkl')
    print("✅ Saved: breakeven_time_model_v1.pkl")
else:
    print(f"\n✅ R²={r2:.4f} - acceptable!")
    joblib.dump(be_model, '../models/breakeven_time_model_v1.pkl')
    print("✅ Saved: breakeven_time_model_v1.pkl")

print("\n" + "="*60)
print("💰 BREAK-EVEN MODEL COMPLETE!")
print("="*60)


💰 MODEL 5: BREAK-EVEN TIME (UPDATED)

📊 Preparing data...
   Samples: 43265 ✅
   Features: 189
   Target range: 1.0 - 84.0 months
   Target mean: 31.1 months

🥊 Training XGBoost...
✅ Training complete!

🏆 BREAK-EVEN TIME RESULTS:
   R² Score: 0.9999
   MAE:      0.15 months
   RMSE:     0.25 months

🔍 TOP 15 FEATURES:
   market_saturation                   0.4524
   target_breakeven_time_v2            0.2794
   product_type_app                    0.1401
   category_product_count              0.0339
   content_rating                      0.0189
   num_founders                        0.0070
   business_model_encoded              0.0057
   desc_5                              0.0047
   funding_per_round                   0.0042
   product_type_digital_app            0.0038
   desc_10                             0.0036
   desc_16                             0.0036
   team_size                           0.0034
   product_type_tool                   0.0033
   description_word_count           

In [19]:
print("🔪 EMERGENCY FIX - Dropping target from features...")

# Add target to leak list
BREAKEVEN_LEAKS_FIXED = BREAKEVEN_LEAKS + ['target_breakeven_time_v2']

# Prepare data
X_be = df.drop(columns=BREAKEVEN_LEAKS_FIXED, errors='ignore')
y_be = df['target_breakeven_time_v2']

mask = y_be.notna()
X_be = X_be[mask]
y_be = y_be[mask]

print(f"   Samples: {len(X_be)}")
print(f"   Features: {X_be.shape[1]}")

# Train
X_train, X_test, y_train, y_test = train_test_split(
    X_be, y_be, test_size=0.2, random_state=42
)

be_model = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.015,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0.2,
    reg_lambda=1.0,
    random_state=42
)

be_model.fit(X_train, y_train)

# Evaluate
y_pred = be_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n🏆 FIXED BREAK-EVEN:")
print(f"   R² Score: {r2:.4f}")
print(f"   MAE:      {mae:.2f} months")
print(f"   RMSE:     {rmse:.2f} months")

# Top features
imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': be_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(f"\n🔍 TOP 10:")
for i, row in imp.head(10).iterrows():
    print(f"   {row['Feature']:<35} {row['Importance']:.4f}")

if r2 > 0.75:
    print(f"\n⚠️ Still {r2:.4f} - more leaks")
elif r2 > 0.30:
    print(f"\n✅ GOOD! {r2:.4f}")
    joblib.dump(be_model, '../models/breakeven_time_model_v1_final.pkl')
    print("✅ Saved!")
else:
    print(f"\n✅ {r2:.4f} acceptable")
    joblib.dump(be_model, '../models/breakeven_time_model_v1_final.pkl')
    print("✅ Saved!")


🔪 EMERGENCY FIX - Dropping target from features...
   Samples: 43265
   Features: 188

🏆 FIXED BREAK-EVEN:
   R² Score: 0.5964
   MAE:      11.63 months
   RMSE:     16.52 months

🔍 TOP 10:
   market_saturation                   0.2800
   product_type_app                    0.1208
   is_premium                          0.0580
   category_encoded                    0.0380
   desc_5                              0.0267
   product_type_digital_app            0.0133
   desc_56                             0.0132
   price_vs_category                   0.0109
   content_rating                      0.0077
   is_free                             0.0076

✅ GOOD! 0.5964
✅ Saved!


In [23]:
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import joblib

print("\n🥞 Training STACKED Break-Even Model...")

# ======================================================
# Base Model 1 — XGBoost (your tuned config)
# ======================================================
xgb_be = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.015,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0.2,
    reg_lambda=1.0,
    random_state=42
)

# ======================================================
# Base Model 2 — CatBoost
# ======================================================
cat_be = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.02,
    depth=8,
    l2_leaf_reg=5,
    bagging_temperature=0.5,
    random_strength=0.8,
    border_count=128,
    random_state=42,
    verbose=False
)

# ======================================================
# Meta Learner — Ridge
# ======================================================
ridge_meta = Ridge(alpha=1.0)

# ======================================================
# STACKING REGRESSOR
# ======================================================
stack_be = StackingRegressor(
    estimators=[
        ('xgb', xgb_be),
        ('cat', cat_be)
    ],
    final_estimator=ridge_meta,
    passthrough=True,     # boosty boost (meta sees full features + preds)
    cv=5,
    n_jobs=-1
)

# Train stack
stack_be.fit(X_train, y_train)

# Predict
stack_pred = stack_be.predict(X_test)

# Metrics
r2 = r2_score(y_test, stack_pred)
mae = mean_absolute_error(y_test, stack_pred)
rmse = np.sqrt(mean_squared_error(y_test, stack_pred))

print(f"\n🏆 STACKED BREAK-EVEN RESULTS:")
print(f"   R² Score: {r2:.4f}")
print(f"   MAE:      {mae:.2f} months")
print(f"   RMSE:     {rmse:.2f} months")

# Save
joblib.dump(stack_be, '../models/breakeven_stack_model_v1.pkl')
print("\n💾 Saved: breakeven_stack_model_v1.pkl")



🥞 Training STACKED Break-Even Model...


C:\Users\mahes\anaconda3\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=9.57742e-27): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)



🏆 STACKED BREAK-EVEN RESULTS:
   R² Score: 0.6013
   MAE:      11.55 months
   RMSE:     16.42 months

💾 Saved: breakeven_stack_model_v1.pkl


In [25]:
# joblib.dump(trac_stack, '../models/traction_time_model_stack_v1.pkl')
# joblib.dump(trac_stack, '../models/traction_time_model_stack_v1.pkl')

joblib.dump(stack_be, '../models/breakeven_time_model/bet_v1/breakeven_time_best_stack_model.pkl')
joblib.dump(xgb_be, '../models/breakeven_time_model/bet_v1/breakeven_time_xgb_model.pkl')
joblib.dump(cat_be, '../models/breakeven_time_model/bet_v1/breakeven_time_cat_model.pkl')
joblib.dump(X_train, '../models/breakeven_time_model/bet_v1/feature_names.pkl')
print("Saved!")

print(X_train.shape)

Saved!
(34612, 188)
